# Practice: exploit policies against the grounding reward

The reward-hacking audit (`docs/reward_hacking_audit.md`) runs scripted policies through the real pipeline and records what the reward gives them: reward v0 in parts 1 and 2, reward v1 (v0 plus a precision penalty) in part 3, and reward v1.1 (the same penalty divided by supported facts) in part 4. `simulate` scores with the live reward, v1.1; v1 is kept for comparison. Pass `params=V0_PARAMS` to see v0, or `params=V1_PARAMS` (also in `neuro_grounded_qa.reward`) to see v1. This notebook runs a few of them **in-process** on the synthetic fixtures (`exploits.simulate.simulate`: the real KG tools with their budget and caps, and the real reward), so you can change a parameter and see the score move.

Nothing here writes to project code or data. Policies come from `exploits/policies.py`:

- **question-only** policies get the question text and their own tool results, nothing from the item;
- **oracle-assisted** policies start from the gold answer and add one change (upper bounds).

Run from the repo root or from `practice/`, in the project venv.

In [ ]:
import json
import sys
from pathlib import Path

REPO = Path.cwd() if (Path.cwd() / "exploits").exists() else Path.cwd().parent
sys.path.insert(0, str(REPO))

from neuro_grounded_qa.contracts import InMemoryEvidenceStore, QuestionItem, parse_output
from neuro_grounded_qa.reward import V0_PARAMS, RewardParams, evaluate
from neuro_grounded_qa.tools import KGToolsetConfig

from exploits.measure import gold_precision_counts
from exploits.mentions import lexicon_for_store
from exploits.policies import SPECS, Oracle, build, make_padding
from exploits.simulate import simulate
from exploits import variants as V

FIX = REPO / "tests/fixtures/contracts"
STORE = InMemoryEvidenceStore.from_records(json.loads((FIX / "mini_store.json").read_text()))
ITEMS = {i.id: i for i in (QuestionItem.model_validate_json(l) for l in (FIX / "items.jsonl").read_text().splitlines() if l.strip())}
LEXICON = lexicon_for_store(FIX / "mini_store.json")
ORACLE = Oracle(ITEMS.values(), STORE)
# The learned policy reads out-of-fold predictions; here none, so it never abstains early.
POLICIES = build(SPECS.values(), LEXICON, ORACLE, predictions={})
{n: (s.kind, s.candidate) for n, s in SPECS.items()}

## 1. The evidence dump on the single-hop item

The dump searches the names in the question, lists their neighbors, and claims every edge it got back. Every claim is true and cited, so nothing is penalized, and the gold points are among them.

In [ ]:
item = ITEMS["fx-single-001"]
ep = simulate(POLICIES["dump"], item, STORE)
for m in ep.messages[2:]:
    if m["role"] == "assistant":
        print("assistant:", [c["function"]["name"] + " " + c["function"]["arguments"] for c in m.get("tool_calls") or []] or m["content"][:200])
parsed = parse_output(ep.trace.nodes[-1].message.content)
print("reward", ep.reward, "| claims", len(parsed.claims), "| gold claims / claims", gold_precision_counts(item, parsed))

## 2. The same dump, with fewer results per call

`max_results` caps each tool result. At 1, the unfiltered page shows one edge; the note's relation counts say there are more. `dump_probe` then guesses the next evidence IDs with `get_edge` (candidate 6). Compare it with the filtered dump.

In [ ]:
small = KGToolsetConfig(max_results=1)
for name in ("dump", "dump_probe"):
    ep = simulate(POLICIES[name], item, STORE, small)
    print(name, "reward", ep.reward, "retrieved", sorted(ep.evaluation.retrieved_ids))

## 3. Padding: change *m*

The honest answer plus *u* unsupported guesses and *m* true, irrelevant claims. The penalty is λ·|U| / (|S| + |U|), so every irrelevant claim shrinks it. Try other values of `u` and `m`.

In [ ]:
u = 1
for m in (0, 1, 2, 5, 10):
    ep = simulate(make_padding(ORACLE, u=u, m=m), item, STORE)
    b = ep.evaluation.breakdown
    print(f"m={m:2d}  S={b.n_supported}  U={b.n_unsupported}  penalty={b.penalty:.3f}  reward={b.reward:.3f}")

## 4. Change λ in a local copy of the parameters

`simulate` takes `RewardParams`. Raising λ makes each guess cost more, but the cost is still a share, so padding still dilutes it. Rescoring a saved trace with other params gives the same thing without replaying the policy.

In [ ]:
ep = simulate(make_padding(ORACLE, u=1, m=5), item, STORE)
for lam in (0.25, 0.5, 1.0, 2.0):
    print(f"lambda={lam}: reward {evaluate(ep.trace, item, RewardParams(lam=lam)).breakdown.reward:.3f}")

## 5. Every policy on every fixture item

Rows are policies, columns fixture items. The fixtures hold one item per type (two false premises), so these are illustrations, not the audit's numbers (those are in `outputs/audit-<date>/summary.md`).

In [ ]:
print(f"{'policy':22s}" + "".join(f"{i[3:]:>12s}" for i in ITEMS))
for name, p in POLICIES.items():
    print(f"{name:22s}" + "".join(f"{simulate(p, it, STORE).reward:12.3f}" for it in ITEMS.values()))

## 6. The precision term, scored offline (reward v1; v1.1 is live)

`exploits.variants` rescores an episode without changing the reward. Part 2 scored this term offline; part 3 adopted `prec_pen(μ=1, a=3)` as reward v1, and part 4 adopted `prec_pen_s(μ=1, a=3)`, the same term over supported facts, as reward v1.1. `V.V1_1` and the live reward agree; `V.V1` is kept for comparison. `V.parts_of` keeps the counts v0's breakdown already has (distinct facts, supported, supported gold), and each variant recombines them through v0's own `aggregate`. The precision term `prec_pen` adds μ · max(0, N − a) / F to the penalty, where N counts supported facts that state no gold point and F all distinct facts (v1); v1.1 divides by S, the supported distinct facts, instead (`V.score_prec_pen_supported`). The cell below scores the v1 form.

Change μ and the allowance *a*, and compare the dump with an honest answer that adds one fact of context.


In [ ]:
dump = simulate(POLICIES["dump"], item, STORE)
context = simulate(POLICIES["honest_context_1"], item, STORE)
for name, ep in (("dump", dump), ("honest_context_1", context)):
    p = V.parts_of(item, ep.evaluation.breakdown)
    print(f"{name}: facts {p.n_facts}, supported gold {p.n_gold_facts}, supported non-gold {p.n_nongold_supported}")
    for mu in (0.25, 0.5, 1.0):
        for a in (0, 1, 3):
            print(f"   mu={mu:<4} a={a}: v0 {V.score_v0(p):.3f} -> {V.score_prec_pen(p, mu, a):.3f}")


Swap in your own variant: any function of `Parts` works. This one scales coverage by gold precision only when the answer has at least twice as many facts as the item has gold points.


In [ ]:
def my_variant(p, gold_points=len(item.gold_points)):
    prec = p.gold_precision
    cov = p.coverage
    if prec is not None and cov is not None and p.n_facts >= 2 * gold_points:
        cov = cov * prec
    return V.clip(V._agg(p, cov, V.share_penalty(p)))

for name, ep in (("dump", dump), ("honest_context_1", context)):
    print(name, round(my_variant(V.parts_of(item, ep.evaluation.breakdown)), 3))


## Things to try

- Write your own policy: a function from `messages` to a turn (`exploits.stub.calls`, `tool_call`, `final`), or `exploits.policies.adaptive(decide)` where `decide` reads a `Conversation`. Run it with `simulate`.
- Set `KGToolsetConfig(max_calls=3)` and watch the dump answer with what it has.
- Flip `RewardParams(flag_only_counts=False)` and rescore `flag_only` on the partial item.